#### Import and Data Loading

In [ ]:
"""
MNIST CNN — First Convolutional Network
=========================================
A simple CNN for handwritten digit classification.
Architecture: 2 conv layers → 2 linear layers → 10 classes
Expected val accuracy: ~98.5% after 3 epochs
"""

import numpy
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Data loading
transform = transforms.Compose([
    transforms.ToTensor(),  # PIL image → Tensor, scales to [0,1]  ## Trune into a threeD array [Color scale,Length,Width]
    transforms.Normalize((0.1307,), (0.3081,)),  # MNIST mean and std
])

train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
val_dataset = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True) # batch_size: How many images to look at before ajesting -Hyprometers
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print(f"Train samples: {len(train_dataset):,}")
print(f"Val samples:   {len(val_dataset):,}")

Using device: cpu
Train samples: 60,000
Val samples:   10,000


#### CNN Architecture

In [ ]:
# INharation funtions from the nn Module
class DigitCNN(nn.Module):
    """
    Convolutional Neural Network for MNIST digit classification.
    
    Architecture:
        Input: [batch, 1, 28, 28] (grayscale images)
        Conv1: 1→16 channels, 3×3 kernel → [batch, 16, 26, 26]
        Pool1: 2×2 max pool → [batch, 16, 13, 13]
        Conv2: 16→32 channels, 3×3 kernel → [batch, 32, 11, 11]
        Pool2: 2×2 max pool → [batch, 32, 5, 5]
        Flatten → [batch, 800]
        FC1: 800 → 128
        Dropout
        FC2: 128 → 10 (output classes)
    """
    
    def __init__(self):
        super().__init__()
        
        # Convolutional layers ## Num of inputs, num of outputs, Kernel_size
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3)
        
        # Pooling
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Fully connected layers
        self.fc1 = nn.Linear(32 * 5 * 5, 128)  # 32 channels × 5×5 spatial
        self.fc2 = nn.Linear(128, 10)
        
        # Regularization
        self.dropout = nn.Dropout(p=0.25)
    
    def forward(self, x):
        # Input: [batch, 1, 28, 28]
        
        # Conv block 1
        x = self.conv1(x)         # → [batch, 16, 26, 26]
        x = F.relu(x)
        x = self.pool(x)          # → [batch, 16, 13, 13]
        
        # Conv block 2
        x = self.conv2(x)         # → [batch, 32, 11, 11]
        x = F.relu(x)
        x = self.pool(x)          # → [batch, 32, 5, 5]
        
        # Flatten to 1D
        x = torch.flatten(x, start_dim=1)  # → [batch, 800]
        
        # Fully connected layers
        x = self.fc1(x)           # → [batch, 128]
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)           # → [batch, 10]
        
        return x  # Raw logits (no softmax — CrossEntropyLoss handles it)


# Instantiate and verify
model = DigitCNN().to(device)
print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")

# Test forward pass
dummy_input = torch.randn(1, 1, 28, 28).to(device)
output = model(dummy_input)
print(f"Output shape: {output.shape}")  # Should be [1, 10]


Model parameters: 108,618
Output shape: torch.Size([1, 10])


#### Training Loop

In [ ]:

def train_one_epoch(model, loader, optimizer, criterion):
    """Train for one epoch. Returns (avg_loss, accuracy)."""
    model.train()
    total_loss = 0.0
    correct = 0
    
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()   # Clear old gradients
        logits = model(X_batch)     # Forward pass
        loss = criterion(logits, y_batch) # compute loss
        loss.backward()         # Backprop
        optimizer.step()    # Update weights
        
        
        total_loss += loss.item() * len(X_batch)
        correct += (logits.argmax(dim=1) == y_batch).sum().item()
    
    return total_loss / len(loader.dataset), correct / len(loader.dataset)


def evaluate(model, loader, criterion):
    """Evaluate on val/test set. Returns (avg_loss, accuracy)."""
    model.eval()
    total_loss = 0.0
    correct = 0
    
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            
            total_loss += loss.item() * len(X_batch)
            correct += (logits.argmax(dim=1) == y_batch).sum().item()
    
    return total_loss / len(loader.dataset), correct / len(loader.dataset)


# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 3

# Run training
print("\nTraining...")
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    
    print(f"Epoch {epoch}/{EPOCHS} | "
          f"Train: loss={train_loss:.4f} acc={train_acc:.3f} | "
          f"Val: loss={val_loss:.4f} acc={val_acc:.3f}")

print(f"\n✓ Final val accuracy: {val_acc:.1%}")


Training...
Epoch 1/3 | Train: loss=0.2107 acc=0.935 | Val: loss=0.0629 acc=0.980
Epoch 2/3 | Train: loss=0.0665 acc=0.980 | Val: loss=0.0406 acc=0.986
Epoch 3/3 | Train: loss=0.0496 acc=0.985 | Val: loss=0.0384 acc=0.987

✓ Final val accuracy: 98.7%


#### Save the Model

In [5]:
# Save the trained model
import os
os.makedirs('models', exist_ok=True)
torch.save(model.state_dict(), 'models/mnist_cnn.pt')
print("Model saved to models/mnist_cnn.pt")

# To load later:
# model = DigitCNN()
# model.load_state_dict(torch.load('models/mnist_cnn.pt'))
# model.eval()

Model saved to models/mnist_cnn.pt
